In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
from shapely.geometry import box, Point

datasets = ["", "_CAW009"]

# Read parquet file
#tissue_positions_8 = pd.read_parquet("/home/chananchidas/visium-hd/data/Visium_HD_Liver/binned_outputs/square_008um/spatial/tissue_positions.parquet")
#tissue_positions_16 = pd.read_parquet("/home/chananchidas/visium-hd/data/Visium_HD_Liver/binned_outputs/square_016um/spatial/tissue_positions.parquet")

In [9]:
def get_containing_barcode(dataset, bins=(8, 16),
                           data_path = "/home/chananchidas/visium-hd/data/"):

	# Read parquet files
	tissue_positions_8 = pd.read_parquet(f"{data_path}Visium_HD_Liver{dataset}/binned_outputs/square_{bins[0]:03}um/spatial/tissue_positions.parquet")
	tissue_positions_16 = pd.read_parquet(f"{data_path}Visium_HD_Liver{dataset}/binned_outputs/square_{bins[1]:03}um/spatial/tissue_positions.parquet")
   
	# Read JSON file for 16um
	with open(f"{data_path}Visium_HD_Liver{dataset}/binned_outputs/square_{bins[1]:03}um/spatial/scalefactors_json.json", "r") as file:
		json_data = json.load(file)
	spot_scalefactor_16 = json_data['spot_diameter_fullres']

	# Filter to in_tissue = 1
	tissue_positions_8 = tissue_positions_8[tissue_positions_8['in_tissue'] == 1]
	tissue_positions_16 = tissue_positions_16[tissue_positions_16['in_tissue'] == 1]

	# Create a GeoDataFrame with square geometries for 16um
	tissue_positions_16['geometry'] = tissue_positions_16.apply(
		lambda row: box(
			row['pxl_row_in_fullres'] - (spot_scalefactor_16 / 2),
			row['pxl_col_in_fullres'] - (spot_scalefactor_16 / 2),
			row['pxl_row_in_fullres'] + (spot_scalefactor_16 / 2),
			row['pxl_col_in_fullres'] + (spot_scalefactor_16 / 2)
		),
		axis=1
	)

	# Create a Point geometry for each spot in tissue_positions_8
	tissue_positions_8['geometry'] = tissue_positions_8.apply(
		lambda row: Point(row['pxl_row_in_fullres'], row['pxl_col_in_fullres']),
		axis=1
	)

	# Create GeoDataFrames
	tissue_positions_16_gdf = gpd.GeoDataFrame(tissue_positions_16, geometry='geometry')
	tissue_positions_8_gdf = gpd.GeoDataFrame(tissue_positions_8, geometry='geometry')

	# Spatial join to find which points in tissue_positions_8 are contained in geometries of tissue_positions_16
	joined = gpd.sjoin(tissue_positions_8_gdf[['barcode', 'geometry']], tissue_positions_16_gdf[['barcode', 'geometry']],
					   how='inner', predicate='within', lsuffix='8', rsuffix='16')

	# Add the containing barcode to tissue_positions_8
	tissue_positions_8['containing_barcode'] = joined['barcode_16']

	# Filter out points that are not contained in any geometry
	contained_points = tissue_positions_8[tissue_positions_8['containing_barcode'].notnull()]
	print(f"All points in {bins[0]}um are contained in {bins[1]}um:", contained_points.shape[0] == tissue_positions_8.shape[0])

	# Save tissue_positions_8 with containing_barcode to a new parquet file
	tissue_positions_8.drop(columns=['geometry']).to_parquet(f"/home/chananchidas/visium-hd/visium_hd_liver_combined/rds/tissue_positions_{bins[0]:03}um_{bins[1]:03}um{dataset}.parquet")



In [10]:
#get_containing_barcode(dataset="")
get_containing_barcode(dataset="_CAW009", bins=(8,16))

All points in 8um are contained in 16um: True


In [ ]:
# Read JSON file for 16um
with open("/home/chananchidas/visium-hd/data/Visium_HD_Liver/binned_outputs/square_016um/spatial/scalefactors_json.json", "r") as file:
	json_data = json.load(file)
spot_scalefactor_16 = json_data['spot_diameter_fullres']

In [20]:
# Filter to in_tissue = 1
tissue_positions_8 = tissue_positions_8[tissue_positions_8['in_tissue'] == 1]
tissue_positions_16 = tissue_positions_16[tissue_positions_16['in_tissue'] == 1]

In [21]:


# Create a GeoDataFrame with square geometries
tissue_positions_16['geometry'] = tissue_positions_16.apply(
    lambda row: box(
        row['pxl_row_in_fullres'] - (spot_scalefactor_16/2),
        row['pxl_col_in_fullres'] - (spot_scalefactor_16/2),
        row['pxl_row_in_fullres'] + (spot_scalefactor_16/2),
        row['pxl_col_in_fullres'] + (spot_scalefactor_16/2)
    ),
    axis=1
)

#tissue_positions_8_gdf = gpd.GeoDataFrame(tissue_positions_8, geometry='geometry')

In [22]:


# Create a Point geometry for each spot in tissue_positions_8
tissue_positions_8['geometry'] = tissue_positions_8.apply(
    lambda row: Point(row['pxl_row_in_fullres'], row['pxl_col_in_fullres']),
    axis=1
)

In [26]:
# Create a GeoDataFrame for tissue_positions_16 if not already done
tissue_positions_16_gdf = gpd.GeoDataFrame(tissue_positions_16, geometry='geometry')

# Spatial join to find which points in tissue_positions_8 are contained in geometries of tissue_positions_16
tissue_positions_8_gdf = gpd.GeoDataFrame(tissue_positions_8, geometry='geometry')
joined = gpd.sjoin(tissue_positions_8_gdf[['barcode', 'geometry']], tissue_positions_16_gdf[['barcode', 'geometry']],
                   how='inner', predicate='within', lsuffix='8', rsuffix='16')

# Add the containing barcode to tissue_positions_8
tissue_positions_8['containing_barcode'] = joined['barcode_16']

# Filter out points that are not contained in any geometry
contained_points = tissue_positions_8[tissue_positions_8['containing_barcode'].notnull()]
print("All points in 8um are contained in 16um:", contained_points.shape[0] == tissue_positions_8.shape[0])

All points in 8um are contained in 16um: True


In [ ]:
tissue_positions_8

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,geometry,containing_barcode
0,s_008um_00000_00000-1,1,0,0,3681.497801,37341.631810,POINT (3681.498 37341.632),s_016um_00000_00000-1
1,s_008um_00000_00001-1,1,0,1,3682.793909,37305.319533,POINT (3682.794 37305.32),s_016um_00000_00000-1
2,s_008um_00000_00002-1,1,0,2,3684.090018,37269.007258,POINT (3684.09 37269.007),s_016um_00000_00001-1
3,s_008um_00000_00003-1,1,0,3,3685.386127,37232.694984,POINT (3685.386 37232.695),s_016um_00000_00001-1
4,s_008um_00000_00004-1,1,0,4,3686.682235,37196.382713,POINT (3686.682 37196.383),s_016um_00000_00002-1
...,...,...,...,...,...,...,...,...
701025,s_008um_00836_00457-1,1,836,457,34631.455044,21828.810137,POINT (34631.455 21828.81),s_016um_00418_00228-1
701026,s_008um_00836_00458-1,1,836,458,34632.750253,21792.498692,POINT (34632.75 21792.499),s_016um_00418_00229-1
701027,s_008um_00836_00459-1,1,836,459,34634.045462,21756.187249,POINT (34634.045 21756.187),s_016um_00418_00229-1
701096,s_008um_00836_00528-1,1,836,528,34723.414682,19250.702690,POINT (34723.415 19250.703),s_016um_00418_00264-1


In [ ]:
# Remove the geometry column
tissue_positions_8 = tissue_positions_8.drop(columns=['geometry'])

# Save tissue_positions_8 with containing_barcode to a new parquet file
tissue_positions_8.to_parquet("/home/chananchidas/visium-hd/data/Visium_HD_Liver/binned_outputs/square_008um/spatial/tissue_positions_with_containing_barcode.parquet")